# NB03 — Full-mosaic M2 Musanga probability inference

Frozen NB02B global M2 three-fold ensemble, native RGB only. 256×256 input patches as trained, stride 128, globally anchored cosine feather blending, independent seam preflight, all audited mosaics including MOS3–4. Restartable stripes and protected existing products. No training. Ensemble is not itself cross-validated; do not attribute NB02B fold CV metrics to this ensemble.


## 01 — Input audit and frozen-checkpoint verification


In [ ]:
%pip -q install rasterio segmentation-models-pytorch pandas matplotlib
import json,hashlib,os,shutil,time,math,warnings,gc
from pathlib import Path
import numpy as np,pandas as pd,rasterio,torch
from rasterio.windows import Window
from rasterio.errors import NodataShadowWarning
from tqdm.auto import tqdm
from IPython.display import display
from google.colab import drive
warnings.filterwarnings("once",category=NodataShadowWarning)
if not Path("/content/drive/MyDrive").exists(): drive.mount("/content/drive")
ROOT=next((p for p in (Path("/content/drive/MyDrive/Colab Notebooks/Musanga/Final"),Path("/content/drive/MyDrive/Musanga/Final")) if p.is_dir()),None)
assert ROOT is not None,"Missing Musanga/Final on Drive"
AUDIT=ROOT/"Outputs/V3/00_input_audit/00_input_manifest.json"
BASE=ROOT/"Outputs/V3/02B_spatial3fold_no_MOS34"
assert AUDIT.exists() and BASE.exists()
A=json.loads(AUDIT.read_text())
assert A.get("schema_version")=="v3_input_audit_5_nadine_circles"
assert torch.cuda.is_available(),"Choose a Colab GPU runtime"
import segmentation_models_pytorch as smp

class ResNet18UNet(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.net=smp.Unet(encoder_name="resnet18",encoder_weights=None,in_channels=3,classes=1)
    def forward(self,x): return self.net(x)[:,0]

MOSAICS={m["file"]:Path(m["path"]) for m in A["mosaics"]}
assert len(MOSAICS)>=6,("Expected all audited mosaics, including MOS3–4",list(MOSAICS))
assert all(p.exists() for p in MOSAICS.values())
INCLUDE_LUKI=False  # external to NB00; may enable explicitly
if INCLUDE_LUKI:
    source=ROOT.parent/"data/Geo_Ref/Luki.tif"
    assert source.exists(),source
    MOSAICS["Luki.tif"]=source
RUN_MOSAICS=list(MOSAICS)  # choose a subset to test before full production
assert set(RUN_MOSAICS).issubset(MOSAICS)

ref_file=BASE/"02B_latest_reference_version.json"
assert ref_file.exists()
ref=json.loads(ref_file.read_text())
assert ref["circles"]==30 and ref["sha256"]
scored=pd.read_csv(BASE/"02B_spatialCV_all30_paired.csv")
scored=scored[scored.model=="M2"]
assert len(scored)==30 and scored.circle_id.is_unique
def sha_file(p):
    h=hashlib.sha256()
    with open(p,"rb") as f:
        for piece in iter(lambda:f.read(8*1024*1024),b""): h.update(piece)
    return h.hexdigest()
CHECKPOINTS=[]
for fold in (0,1,2):
    matches=sorted((BASE/f"fold_{fold}").glob("M2_resnet18_spatialCV_*.pt"))
    assert len(matches)==1,("Resolve ambiguous/missing M2 checkpoint manually",fold,matches)
    p=matches[0]
    CHECKPOINTS.append(dict(fold=fold,path=str(p),sha256=sha_file(p)))
display(pd.DataFrame(CHECKPOINTS))
print("Audited input mosaics:",list(MOSAICS))
print("Frozen 3-fold M2 ensemble. No final refit, no illumination changes.")
# NB02B consumes 0–1 RGB without additional ImageNet mean/std normalization.
TILE=256;STEP=128;CORE=512;STRIPE=1024;BATCH=12
SCALE=10000;NODATA=65535;EDGE_FLOOR=0.02
PIPELINE="NB03_frozen_M2_3fold_native256_stride128_cosine_v1"
assert STEP==TILE//2 and CORE%STEP==0 and STRIPE%CORE==0
OUT=ROOT/"Outputs/V3/03_M2_full_mosaic"
MAPS=OUT/"probability_maps";PARTS=OUT/"stripes"
MAPS.mkdir(parents=True,exist_ok=True);PARTS.mkdir(parents=True,exist_ok=True)
SCRATCH=Path("/content/nb03_scratch");SCRATCH.mkdir(exist_ok=True)
ENSEMBLE_SHA=hashlib.sha256("".join(z["sha256"] for z in CHECKPOINTS).encode()).hexdigest()


## 02 — Native-resolution 256-pixel inference with globally aligned cosine blending
Every mosaic pixel receives contributions from the same global tile lattice regardless of output block or stripe position. Three fold probabilities are averaged, never their logits. This is computationally expensive but avoids spatial-window mismatch from V2.


In [ ]:
MODELS=[]
for x in CHECKPOINTS:
    model=ResNet18UNet().cuda()
    model.load_state_dict(torch.load(x["path"],map_location="cpu",weights_only=True),strict=True)
    model.eval()
    MODELS.append(model)
assert len(MODELS)==3
u=np.arange(TILE,dtype="float32")+.5
u=EDGE_FLOOR+(1-EDGE_FLOOR)*np.sin(np.pi*u/TILE)**2
WEIGHT=(u[:,None]*u[None,:]).astype("float32")
assert WEIGHT.min()>0
def valid_rgb(raw):
    # MATCH NB02B: masked RGB bands, excluding exact all-black, all-white, blue.
    a=raw.filled(0).astype("float32")
    v=~np.any(np.ma.getmaskarray(raw),axis=0)
    v &= np.any(a!=0,axis=0)&np.any(a!=255,axis=0)
    v &= ~((a[0]==0)&(a[1]==0)&(a[2]==255))
    return v

def starts(lo,hi):
    return list(range((lo-TILE+1)//STEP*STEP,(hi-1)//STEP*STEP+1,STEP))

def infer_rect(src,r0,c0,h,w):
    """Globally anchored overlapping 256-pixel windows; no core-dependent seams."""
    assert 0<=r0<src.height and 0<=c0<src.width
    assert 0<h<=src.height-r0 and 0<w<=src.width-c0
    center=src.read([1,2,3],window=Window(c0,r0,w,h),masked=True)
    valid=valid_rgb(center)
    if not valid.any():
        return np.full((h,w),np.nan,"float32"),valid
    rs=starts(r0,r0+h);cs=starts(c0,c0+w)
    top=min(rs);left=min(cs);bottom=max(rs)+TILE;right=max(cs)+TILE
    ext=np.zeros((3,bottom-top,right-left),dtype="uint8")
    rr0=max(0,top);cc0=max(0,left)
    rr1=min(bottom,src.height);cc1=min(right,src.width)
    if rr1>rr0 and cc1>cc0:
        src_data=src.read([1,2,3],window=Window(cc0,rr0,cc1-cc0,rr1-rr0),masked=True)
        ext[:,rr0-top:rr1-top,cc0-left:cc1-left]=src_data.filled(0)
    total=np.zeros((h,w),dtype="float64")
    weights_total=np.zeros((h,w),dtype="float64")
    patches=[];tasks=[]
    def flush():
        if not patches:return
        batch=torch.from_numpy(np.stack(patches).astype("float32")/255.).cuda()
        ensemble=None
        with torch.inference_mode():
            for model in MODELS:
                with torch.autocast("cuda",dtype=torch.float16):
                    prediction=torch.sigmoid(model(batch)).float()
                ensemble=prediction if ensemble is None else ensemble+prediction
            predictions=(ensemble/len(MODELS)).cpu().numpy()
        for (tr,tc,cr0,cr1,ccr0,ccr1),prob in zip(tasks,predictions):
            pr0=cr0-(tr-r0);pc0=ccr0-(tc-c0)
            hh=cr1-cr0;ww=ccr1-ccr0
            wwgt=WEIGHT[pr0:pr0+hh,pc0:pc0+ww]
            pp=prob[pr0:pr0+hh,pc0:pc0+ww]
            total[cr0:cr1,ccr0:ccr1]+=wwgt*pp
            weights_total[cr0:cr1,ccr0:ccr1]+=wwgt
        patches.clear();tasks.clear()
    for tr in rs:
        for tc in cs:
            cr0=max(0,tr-r0);cr1=min(h,tr+TILE-r0)
            ccr0=max(0,tc-c0);ccr1=min(w,tc+TILE-c0)
            if cr0>=cr1 or ccr0>=ccr1:continue
            if not valid[cr0:cr1,ccr0:ccr1].any():continue
            patch=ext[:,tr-top:tr-top+TILE,tc-left:tc-left+TILE]
            assert patch.shape==(3,TILE,TILE)
            patches.append(patch);tasks.append((tr,tc,cr0,cr1,ccr0,ccr1))
            if len(patches)>=BATCH:flush()
    flush()
    assert np.all(weights_total[valid]>0),("Missing tile weight",r0,c0)
    p=np.full((h,w),np.nan,"float32")
    p[valid]=(total[valid]/weights_total[valid]).astype("float32")
    assert np.all(np.isfinite(p[valid])) and np.all((p[valid]>=0)&(p[valid]<=1))
    return p,valid

def to_u16(prob,valid):
    arr=np.full(prob.shape,NODATA,dtype="uint16")
    arr[valid]=np.rint(prob[valid]*SCALE).astype("uint16")
    assert np.array_equal(arr!=NODATA,valid)
    return arr
